# `run_results.csv` quality and benchmark-readiness review

## TL;DR

The collector contains one complete 14-configuration sweep with internally consistent core fields, but it is not yet sufficient for model selection. All runs share one stock, one seed, and one temporal split; the trade backtest has only 40 observations per model; and several essential provenance, baseline, uncertainty, calibration, and cost fields are absent from the flat CSV.

## Context and methods

This notebook audits `results/runs/run_results.csv` at the intended grain of one row per completed model/backend × ticker × seed × configuration. It trims CSV padding in memory, checks identity, completeness, range and cross-field consistency, ranks the recorded test metrics, and reads the matching `result.json` payloads only to identify useful fields that exist but are not promoted into the CSV.

In [1]:
from pathlib import Path
import json
import numpy as np
import pandas as pd

ROOT = Path.cwd()
CSV_PATH = ROOT / 'results' / 'runs' / 'run_results.csv'
raw = pd.read_csv(CSV_PATH)
raw_columns = raw.columns.tolist()
df = raw.copy()
df.columns = df.columns.str.strip()
for column in df.select_dtypes(include='object'):
    df[column] = df[column].map(lambda x: x.strip() if isinstance(x, str) else x).replace('', pd.NA)
print(f'Rows: {len(df):,}; columns: {len(df.columns):,}')
print(f'Headers requiring whitespace trimming: {sum(c != c.strip() for c in raw_columns)}/{len(raw_columns)}')
df[['model_family', 'model_label', 'ticker', 'seed', 'status']].head()

Rows: 14; columns: 60
Headers requiring whitespace trimming: 60/60


,model_family,model_label,ticker,seed,status
0,recurrent,lstm,AAPL,42,success
1,recurrent,gru,AAPL,42,success
2,classical,arima,AAPL,42,success
3,classical,random_forest,AAPL,42,success
4,panel,panel_gru,AAPL,42,success


## Structural and validity checks

In [2]:
identity_key = [
    'model_label', 'ticker', 'seed', 'universe_id', 'interval',
    'prediction_window', 'horizon_bars', 'graph_mode', 'graph_embed',
    'graph_ablation', 'ablate_feature',
]
start = pd.to_datetime(df['timestamp_start'])
end = pd.to_datetime(df['timestamp_end'])
bounded = [
    'threshold_operational', 'threshold_validation_macro_f1',
    'accuracy_dense', 'f1_positive_dense', 'macro_f1_dense',
    'roc_auc_dense', 'average_precision_dense', 'accuracy_fixed_05',
    'f1_positive_fixed_05', 'macro_f1_fixed_05',
    'accuracy_trade_aligned', 'f1_positive_trade_aligned',
    'macro_f1_trade_aligned', 'roc_auc_trade_aligned',
    'average_precision_trade_aligned', 'hit_rate',
]
checks = pd.Series({
    'all_status_success': df['status'].eq('success').all(),
    'run_id_unique': df['run_id'].is_unique,
    'intended_key_unique': not df.duplicated(identity_key).any(),
    'duration_matches_timestamps': np.allclose((end-start).dt.total_seconds(), df['duration_sec']),
    'operational_threshold_matches_validation_choice': np.allclose(
        df['threshold_operational'], df['threshold_validation_macro_f1']
    ),
    'all_bounded_metrics_in_0_1': not (((df[bounded] < 0) | (df[bounded] > 1)).any().any()),
    'losses_nonnegative': not (df[['validation_loss', 'test_loss']] < 0).any().any(),
    'drawdown_in_minus1_0': df['max_drawdown'].between(-1, 0).all(),
    'final_equity_positive': df['final_equity'].gt(0).all(),
})
checks.to_frame('passed')

,passed
all_status_success,True
run_id_unique,True
intended_key_unique,True
duration_matches_timestamps,True
operational_threshold_matches_validation_choice,True
all_bounded_metrics_in_0_1,True
losses_nonnegative,True
drawdown_in_minus1_0,True
final_equity_positive,True


In [3]:
missing = df.isna().sum().loc[lambda s: s.gt(0)].sort_values(ascending=False)
display(missing.to_frame('missing_rows'))
print('Dense denominators:', df['n_predictions_dense'].value_counts().to_dict())
print('Trade denominators:', df['n_trades'].value_counts().to_dict())
print('Stocks:', df['ticker'].unique().tolist(), 'Seeds:', df['seed'].unique().tolist())

,missing_rows
job_id,14
error_message,14
queue_run_id,14
graph_backend,6
graph_model,6
energy_wh,2
average_power_w,2
peak_gpu_memory_mb,2
energy_per_sample_wh,2
total_parameters,1


Dense denominators: {970: 14}
Trade denominators: {40: 14}
Stocks: ['AAPL'] Seeds: [42]


## Recorded performance is weak and rankings are unstable by objective

In [4]:
ranked = df.sort_values('macro_f1_dense', ascending=False).copy()
ranked['delta_macro_vs_fixed_05'] = ranked['macro_f1_dense'] - ranked['macro_f1_fixed_05']
ranked['macro_rank'] = ranked['macro_f1_dense'].rank(method='min', ascending=False).astype(int)
ranked['sharpe_rank'] = ranked['sharpe'].rank(method='min', ascending=False).astype(int)
display(ranked[[
    'macro_rank', 'model_label', 'macro_f1_dense', 'accuracy_dense',
    'roc_auc_dense', 'average_precision_dense', 'threshold_operational',
    'delta_macro_vs_fixed_05', 'sharpe', 'sharpe_rank', 'final_equity',
]].reset_index(drop=True))
print('Dense ROC-AUC > 0.5:', int(df['roc_auc_dense'].gt(0.5).sum()), 'of', len(df))
print('Dense macro-F1 > 0.5:', int(df['macro_f1_dense'].gt(0.5).sum()), 'of', len(df))
print('Threshold selection improved test macro-F1:', int(ranked['delta_macro_vs_fixed_05'].gt(0).sum()), 'of', len(df))
print('Spearman macro-F1 vs Sharpe:', round(df[['macro_f1_dense','sharpe']].corr(method='spearman').iloc[0,1], 3))

,macro_rank,model_label,macro_f1_dense,accuracy_dense,roc_auc_dense,average_precision_dense,threshold_operational,delta_macro_vs_fixed_05,sharpe,sharpe_rank,final_equity
0,1,panel_gru,0.528853,0.528866,0.511647,0.591501,0.488088,0.073491,-0.000324,6,0.975748
1,2,arima,0.486830,0.491753,0.494017,0.616006,0.499667,-0.004555,0.597421,4,1.078065
2,3,stgnn+gcn,0.454832,0.498969,0.434370,0.517804,0.496235,-0.012364,0.738120,3,1.104157
3,4,stgnn+graphsage,0.443686,0.444330,0.402809,0.512953,0.505515,0.012509,-0.499262,7,0.898023
4,5,gat,0.435968,0.458763,0.475967,0.550651,0.505879,-0.022636,-3.240372,14,0.588236
5,6,gru,0.427520,0.456701,0.382235,0.501088,0.503836,-0.005415,-0.635145,8,0.877749
6,7,lstm,0.417039,0.437113,0.421534,0.528192,0.449740,0.110360,1.375826,1,1.225195
7,8,graphsage,0.416171,0.418557,0.384494,0.493996,0.532125,-0.030024,-2.014133,11,0.702859
8,9,panel_lstm,0.398941,0.455670,0.486936,0.596209,0.474619,0.091384,-0.987891,10,0.827758
9,10,nnconv,0.391814,0.404124,0.419443,0.501223,0.514108,-0.052487,-2.275517,13,0.675451


Dense ROC-AUC > 0.5: 1 of 14
Dense macro-F1 > 0.5: 1 of 14
Threshold selection improved test macro-F1: 6 of 14
Spearman macro-F1 vs Sharpe: 0.266


## Useful split and validation fields exist in payloads but are missing from the CSV

In [5]:
payloads = {}
for path in (ROOT / 'results' / 'runs').glob('**/result.json'):
    payload = json.loads(path.read_text(encoding='utf-8'))
    payloads[payload['run_id']] = payload

payload_rows = []
for row in df.itertuples(index=False):
    payload = payloads[row.run_id]
    validation = payload.get('metadata', {}).get('validation_selection', {})
    quality = payload.get('data_quality', {})
    payload_rows.append({
        'model_label': row.model_label,
        'validation_macro_f1_at_selected_threshold': validation.get('macro_f1_at_selected_threshold'),
        'test_macro_f1': row.macro_f1_dense,
        'validation_minus_test': validation.get('macro_f1_at_selected_threshold') - row.macro_f1_dense,
        'validation_n': validation.get('n_predictions'),
        'train_end': quality.get('train_end_date'),
        'validation_start': quality.get('validation_start_date'),
        'validation_end': quality.get('validation_end_date'),
        'test_start': quality.get('test_start_date'),
        'embargo_bars': quality.get('embargo_bars'),
        'loaded_tickers': quality.get('raw_loaded_count'),
    })
payload_df = pd.DataFrame(payload_rows).sort_values('test_macro_f1', ascending=False)
display(payload_df)
print('Mean validation minus test macro-F1:', round(payload_df['validation_minus_test'].mean(), 3))
print('Range:', tuple(round(x, 3) for x in (payload_df['validation_minus_test'].min(), payload_df['validation_minus_test'].max())))

,model_label,validation_macro_f1_at_selected_threshold,test_macro_f1,validation_minus_test,validation_n,train_end,validation_start,validation_end,test_start,embargo_bars,loaded_tickers
4,panel_gru,0.497413,0.528853,-0.031440,944,2025-05-20T13:30:00,2025-05-23T16:30:00,2025-12-16T14:30:00,2025-12-19T17:30:00,24,50
2,arima,0.504435,0.486830,0.017604,944,2025-05-20T13:30:00,2025-05-23T16:30:00,2025-12-16T14:30:00,2025-12-19T17:30:00,24,50
10,stgnn+gcn,0.482002,0.454832,0.027170,944,2025-05-20T13:30:00,2025-05-23T16:30:00,2025-12-16T14:30:00,2025-12-19T17:30:00,24,50
11,stgnn+graphsage,0.380357,0.443686,-0.063329,944,2025-05-20T13:30:00,2025-05-23T16:30:00,2025-12-16T14:30:00,2025-12-19T17:30:00,24,50
7,gat,0.442645,0.435968,0.006677,944,2025-05-20T13:30:00,2025-05-23T16:30:00,2025-12-16T14:30:00,2025-12-19T17:30:00,24,50
1,gru,0.467577,0.427520,0.040057,944,2025-05-20T13:30:00,2025-05-23T16:30:00,2025-12-16T14:30:00,2025-12-19T17:30:00,24,50
0,lstm,0.478852,0.417039,0.061813,944,2025-05-20T13:30:00,2025-05-23T16:30:00,2025-12-16T14:30:00,2025-12-19T17:30:00,24,50
9,graphsage,0.468566,0.416171,0.052395,944,2025-05-20T13:30:00,2025-05-23T16:30:00,2025-12-16T14:30:00,2025-12-19T17:30:00,24,50
5,panel_lstm,0.447349,0.398941,0.048408,944,2025-05-20T13:30:00,2025-05-23T16:30:00,2025-12-16T14:30:00,2025-12-19T17:30:00,24,50
8,nnconv,0.468745,0.391814,0.076931,944,2025-05-20T13:30:00,2025-05-23T16:30:00,2025-12-16T14:30:00,2025-12-19T17:30:00,24,50


Mean validation minus test macro-F1: 0.063
Range: (np.float64(-0.063), np.float64(0.231))


## Takeaways

1. **Collector health:** the sweep is complete and internally coherent, but whitespace-padded headers and values should be fixed at write time.
2. **Benchmark readiness:** one stock × one seed × one split is a smoke test, not a defensible comparison.
3. **Primary missing evidence:** split dates and embargo, class prevalence/confusion counts, validation score at the selected threshold, calibration, naive and buy-and-hold baselines, transaction costs/slippage, uncertainty, code/data provenance, and failure/attempt accounting.
4. **Interpretation:** Panel GRU leads dense macro-F1 (0.529), while LSTM leads Sharpe (1.376); these rankings conflict and are too underpowered to support selection.
5. **Next benchmark gate:** run multiple seeds, stocks, and rolling temporal folds, retain an untouched final holdout, and report paired uncertainty before declaring a winner.